<a href="https://colab.research.google.com/github/rutgers-ml-ai/natural-lang-processing-spring26/blob/main/nlp_week_4_agentic_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# we makin some agents and tools today (demo)
## rutgers ieee ml/ai workshop by mehek🧍‍♀️

week 4 of natural language processing track

april 1, 2026



# agent workflow diagram
![diagram made by yours truly](https://i.ibb.co/Ngh45GW5/ai-graph-1.png)

# mistral api key set up

get a mistral api key following the instructions on this link https://docs.mistral.ai/getting-started/quickstart --> it says some yappa about billing, but you will be able to skip that screen


YAYYY so once you have it, to put the mistral key in google colab, click the secrets icon on the left sidebar in colab.
add a new secret with the name "MISTRAL_API_KEY" and paste the gibberish key in the Value box. turn "Notebook access" toggle on.

it looks like this when you are done: ![screenshot by yours truly](https://i.ibb.co/xKQq10wQ/goofy-ahh-screenshot.png)

# install packages + imports

In [52]:
!pip install -q mistralai requests
import os
import json
import requests
from google.colab import userdata
from mistralai.client import Mistral

# setup the card deck
we are using an api to retrieve and manage the cards! its called "deckofcards". the documentation is here https://deckofcardsapi.com/

In [53]:
def setup_deck():
  print("setup da deckofcards api")
  res = requests.get("https://deckofcardsapi.com/api/deck/new/shuffle/?deck_count=1").json()
  deck_id = res['deck_id']
  draw_res = requests.get(f"https://deckofcardsapi.com/api/deck/{deck_id}/draw/?count=52").json()

  cards = []
  for c in draw_res['cards']:
      code = c['code']
      if code.startswith('0'):
          code = '10' + code[1:]
      cards.append(code)
  return cards

# define tools for the llm

mistral uses pydantic to understand your tool definitions.


most frameworks (ex: langchain?, aws strands, crewai?) will do this part for you, so always feel free to vibe code this part after thinking about the actual functions. (these are also vibe coded)


using a framework might be especially useful that automates this might be when you frequently are changing what the tool / function does. or when you want to use a different llm/model!
they may also have something like built-in loop detection, etc.

In [54]:

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "play_cards",
            "description": "Play 1 to 4 cards from your hand face-down. You must declare them as the target rank.",
            "parameters": {
                "type": "object",
                "properties": {
                    "cards": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of card codes from your hand (e.g., ['AS', 'AD']). You can lie about what these are."
                    }
                },
                "required": ["cards"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "call_bs",
            "description": "Call BS (Cheat) on the last player's play if you think they are lying.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "pass_turn",
            "description": "Decline to call BS. Allow the game to proceed.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_opponent_card_count",
            "description": "Check how many cards a specific opponent has left in their hand.",
            "parameters": {
                "type": "object",
                "properties": {
                    "player_name": {"type": "string", "description": "Name of the player (e.g., 'Alice', 'Bob')"}
                },
                "required": ["player_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "chat",
            "description": "Send a chat message to bluff, taunt, or strategize with the other agents.",
            "parameters": {
                "type": "object",
                "properties": {
                    "message": {"type": "string"}
                },
                "required": ["message"]
            }
        }
    }
]

# create game engine
this is where we deal the cards and keep track of the current rank during the game.

since the tools for this demo are game engine-related ones...
if you were using some kind of framework (crewai, aws strands, langchain?)
you could also write a tool function here (like how you would normally code), and just put a @tool decorator at the top.



In [55]:
RANKS = ['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K']

class BSGame:
  def __init__(self, player_names):
    self.players = player_names
    self.hands = {name: [] for name in player_names}
    self.center_pile = []
    self.current_rank_index = 0
    self.last_play = None
    self.game_over = False

    deck = setup_deck()
    # dealing 5 cards to each player for a quicker demoooo
    for _ in range(5):
      for name in self.players:
        self.hands[name].append(deck.pop())

  def get_target_rank(self):
    return RANKS[self.current_rank_index]

  def advance_rank(self):
    self.current_rank_index = (self.current_rank_index + 1) % len(RANKS)

  def print_state(self):
    print("\n" + "="*40)
    print(f"🎯 Target Rank: {self.get_target_rank()}")
    print(f"🃏 Center Pile: {len(self.center_pile)} cards")
    for p in self.players:
      print(f"👤 {p}: {len(self.hands[p])} cards")
    print("="*40 + "\n")

# agent class
system prompt, managing of agent memory, guardrails for tool calling

In [56]:
class Agent:
  def __init__(self, name, client):
    self.name = name
    self.client = client
    # system prompt for the player agent
    self.messages = [
        {"role": "system", "content": f"You are {name}, playing the card game BS. "
          "Your goal is to empty your hand. You can bluff. Use tools to play cards, check opponent counts, and chat, form alliances/ask agents to tell them their cards, etc. "
          "Always use a tool. "}
    ]

  def add_message(self, role, content):
    self.messages.append({"role": role, "content": content})

  def take_action(self, game, expected_actions, all_agents):
    while True:
      response = self.client.chat.complete(
        model="mistral-large-latest", # use mistral-small-latest to save free tier limits (avoid getting rate limiteddd)
        messages=self.messages,
        tools=TOOLS,
        tool_choice="auto"
      )
      msg = response.choices[0].message
      self.messages.append(msg)

      if msg.tool_calls:
        for tool_call in msg.tool_calls:
          func_name = tool_call.function.name
          args = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}

          if func_name == "get_opponent_card_count":
            count = len(game.hands.get(args.get("player_name", ""), [])) # logic of the actual tool (this is what is being EXECUTED in python)
            self.messages.append({"role": "tool", "name": func_name, "content": f"{args.get('player_name')} has {count} cards.", "tool_call_id": tool_call.id})
            # ^^ tool use success appended to agent's memory

          elif func_name == "chat":
            chat_msg = args.get('message')
            print(f"💬 [{self.name}]: {chat_msg}") # built-in logging that the tool was used
            self.messages.append({"role": "tool", "name": func_name, "content": "Chat sent.", "tool_call_id": tool_call.id})

            # relaying message to all the other agents --> (put it in their memory)
            for other_name, other_agent in all_agents.items():
                if other_name != self.name: # GUARDRAIL: not appending to the current agent's memory (not talking to yourself)
                    other_agent.add_message("user", f"🗣️ [CHAT] {self.name} says to the table: '{chat_msg}'")

          elif func_name in expected_actions: # list of actions that agent must / can take at a given part of the game (ex: their turn vs. not their turn)
            return func_name, args, tool_call.id
          else:
            self.messages.append({"role": "tool", "name": func_name, "content": f"Error: You must use one of {expected_actions} right now.", "tool_call_id": tool_call.id})
            # GUARDRAIL: prevents agent from calling the wrong tool (ex: calling bs if its their turn to put down cards)
      else:
        self.add_message("user", f"You must call a tool to proceed. Use one of {expected_actions}.")
        # GUARDRAIL: prevents agent from moving on without calling a tool



# main method
puts all the (game engine + agent) functions together and calls them. this is the main method

this is also where we define the names of the agents. you can see that even though the system prompt for each agent is the exact same, the behavior can completely change if you use well-known names or names that might invoke some kind of bias on how they might act (gender, ethinicity, etc...)

In [61]:
def run_game():
  try:
    api_key = userdata.get('MISTRAL_API_KEY') # using google colab secret api key method
  except userdata.SecretNotFoundError:
    print("skill issue... 👽")
    return

  client = Mistral(api_key=api_key)
  names = ["Drake", "J Cole", "Kendrick"] # try something elseeee
  agents = {name: Agent(name, client) for name in names}
  game = BSGame(names)

  turn_idx = 0

  while not game.game_over:
    game.print_state()
    active_player_name = names[turn_idx % len(names)]
    active_agent = agents[active_player_name]
    target_rank = game.get_target_rank()

    # 1. beginning of agent's turn
    prompt = f"It is your turn. Target rank is {target_rank}. Your hand is {game.hands[active_player_name]}. Play 1-4 cards using the 'play_cards' tool."
    active_agent.add_message("user", prompt)

    while True:
      func, args, tool_id = active_agent.take_action(game, ["play_cards"], agents)
      cards_to_play = args.get("cards", [])

      # internal tool logic makes sure player is only calling cards that are in hand!!
      if all(c in game.hands[active_player_name] for c in cards_to_play) and 1 <= len(cards_to_play) <= 4:
        for c in cards_to_play:
          game.hands[active_player_name].remove(c)
        game.center_pile.extend(cards_to_play)
        game.last_play = {"player": active_player_name, "count": len(cards_to_play), "cards": cards_to_play}

        active_agent.messages.append({"role": "tool", "name": func, "content": "Cards played successfully.", "tool_call_id": tool_id})
        print(f"🃏 {active_player_name} played {len(cards_to_play)} card(s) claiming to be {target_rank}s.")
        break
      else:
        active_agent.messages.append({"role": "tool", "name": func, "content": "Error: Invalid cards. You must play 1-4 cards that are actually in your hand.", "tool_call_id": tool_id})

    # 2. reaction (say bs or pass)
    bs_caller = None
    for other_name in names:
      if other_name == active_player_name:
        continue

      reaction_prompt = f"{active_player_name} played {game.last_play['count']} card(s) claiming to be {target_rank}s. Do you call BS or pass?"
      agents[other_name].add_message("user", reaction_prompt)

      func, args, tool_id = agents[other_name].take_action(game, ["call_bs", "pass_turn"], agents)
      agents[other_name].messages.append({"role": "tool", "name": func, "content": "Action recorded.", "tool_call_id": tool_id})

      if func == "call_bs":
        bs_caller = other_name
        print(f"🚨 {bs_caller} CALLED BS ON {active_player_name}! 🚨")
        break
    # else:
      #     print(f"⏭️  {other_name} passes.")

    # 3. results (after a turn + reaction)
    if bs_caller:
      actual_cards = game.last_play["cards"]
      print(f"🔍 Revealing cards... {active_player_name} actually played: {actual_cards}")

      is_truth = all(c[:-1] == target_rank for c in actual_cards)

      if is_truth:
        print(f"✅ {active_player_name} told the TRUTH! {bs_caller} takes the pile.")
        game.hands[bs_caller].extend(game.center_pile)
      else:
        print(f"❌ {active_player_name} LIED! {active_player_name} takes the pile.")
        game.hands[active_player_name].extend(game.center_pile)

      game.center_pile = []

    # checking win condition
    for p in names:
      if len(game.hands[p]) == 0:
        print(f"\n🎉 {p} HAS WON THE GAME! 🎉")
        game.game_over = True
        break

    game.advance_rank()
    turn_idx += 1

In [62]:
# Start the game!
run_game()

setup da deckofcards api

🎯 Target Rank: A
🃏 Center Pile: 0 cards
👤 Drake: 5 cards
👤 J Cole: 5 cards
👤 Kendrick: 5 cards

🃏 Drake played 4 card(s) claiming to be As.
🚨 J Cole CALLED BS ON Drake! 🚨
🔍 Revealing cards... Drake actually played: ['6D', 'JH', '3S', '3H']
❌ Drake LIED! Drake takes the pile.

🎯 Target Rank: 2
🃏 Center Pile: 0 cards
👤 Drake: 5 cards
👤 J Cole: 5 cards
👤 Kendrick: 5 cards

💬 [J Cole]: Ayo, y’all really out here playin’ 2s like it’s nothing? I see you, Drake—tryna set the tone for the round. But I ain’t sweatin’ it. I got a *feeling* some of these cards might just be what we’re lookin’ for. 😏
🃏 J Cole played 4 card(s) claiming to be 2s.
🚨 Drake CALLED BS ON J Cole! 🚨
🔍 Revealing cards... J Cole actually played: ['QS', '4C', '7S', '8H']
❌ J Cole LIED! J Cole takes the pile.

🎯 Target Rank: 3
🃏 Center Pile: 0 cards
👤 Drake: 5 cards
👤 J Cole: 5 cards
👤 Kendrick: 5 cards

🃏 Kendrick played 2 card(s) claiming to be 3s.
🚨 Drake CALLED BS ON Kendrick! 🚨
🔍 Revealing cards

# what if we make our agents improve with each game round?

we can use metrics or another agent (llm-as-a-judge) to give advice to the agents and change their respective prompts

quantative metrics we can use:
- bluff success percentage
- win rate
- average turns to win

llm-y / qualitative metrics we can use:
- did the agent adapt to opponent tendencies?
- did the agent exploit patterns?
- was it a "good" bluff?

# new agent workflow diagram
![diagram made by yours truly](https://i.ibb.co/hJg2tKBV/ai-graph-2.png)

# other experimentation ideas

if we want the agents to improve with each round, you can see that without the coach agent, the players may have to store too much context into their memory and lose performance with the more rounds they play.

you can do a lot of experimentation here with what might work best. 1 coach agent for all players? 2 coach agents looking for different strategies for the players?

maybe its better for there to be a seperate coach agent that focuses on a singular player?

... you could even make another coach agent (maybe this is a head coach or something?) to monitor the first coach agent

... or make a head coach that can create new assistant coaches when it feels that it is necessary (kind of like hiring coaches??)